In [2]:
transcript="""Here is a realistic synthetic transcript tailored for an IIM admission interview. Notice how the panelists quickly steer the conversation away from pure technical jargon and focus heavily on business acumen, macro-level awareness, and behavioral traits—the hallmarks of an MBA interview versus an engineering job interview.

You can use this transcript directly with your LLM evaluator prompt.

### **Synthetic IIM Interview Transcript**

**Interviewer 1 (P1):** Good morning. Please take a seat. Let’s start with a brief introduction.

**Candidate:** Good morning, professors. I am Priyanshu Jain. I am currently in my final year of Computer Science and Engineering at Manipal University Jaipur. Academically, I have maintained a strong track record, and professionally, I’ve interned at Coding Jr and recently started as a Salesforce Trainee at PwC. Outside of academics, I enjoy organizing group travel and following the automotive industry.

**Interviewer 2 (P2):** You have a 9.6 CGPA, a stint at PwC, and I see you've also cleared the pipeline for Fidelity International. That is a very solid engineering career trajectory. Why do you want to derail that to pursue an MBA right now? Why not work for three years first?

**Candidate:** I don't see it as a derailment, sir, but rather a pivot. While my engineering coursework taught me how to build efficient systems, my internships made me realize that I am more fascinated by the business logic driving those systems. Furthermore, observing my father run his electronics and steno-services business has shown me the practical challenges of scaling operations and managing supply chains. I want an MBA to bridge my technical problem-solving skills with formal frameworks in finance and strategy, eventually aiming for a tech-strategy or product management role.

**Interviewer 3 (P3):** Interesting that you mention your father’s business. If you had to consult for his electronics business today to increase his profit margins by 15% over the next year, what two immediate strategies would you implement? And don't just say "build an app."

**Candidate:** *(Pauses to think)* First, I would optimize inventory turnover. Local electronics businesses often tie up capital in slow-moving stock. I would analyze past sales data to shift towards a just-in-time inventory model for high-value items, reducing holding costs. Second, I would bundle the steno-services with hardware sales—for instance, offering discounted maintenance contracts with new equipment purchases to create a recurring revenue stream rather than relying solely on one-off hardware margins.

**P3:** Better than I expected. Let's switch gears. You mentioned an interest in the automotive sector. The government has been heavily pushing for EVs, yet we still see companies launching diesel variants for SUVs, and corporate fleets heavily rely on them. Why does diesel still make economic sense in India?

**Candidate:** It comes down to the Total Cost of Ownership (TCO) over long distances. While EVs are great for urban commuting, corporate fleets and long-haul users require high fuel efficiency under heavy loads and continuous running—something diesel engines excel at. Vehicles like the Hyundai Alcazar diesel, for instance, offer superior highway mileage and torque compared to petrol, making them highly economical for frequent inter-city travel where EV charging infrastructure is still unpredictable.

**P1:** Let's test your academic fundamentals. Since you study Computer Science, you must be familiar with databases. Explain to me—a person with zero IT background—the difference between a relational database like PostgreSQL and a non-relational one like MongoDB. Give me a business analogy.

**Candidate:** Imagine you are running a highly structured corporate office. Every employee must fill out a standard form with exact fields: Name, ID, Department. This strict, table-based structure is a relational database like PostgreSQL. It is rigid but perfect for financial transactions where consistency is key. Now, imagine a creative agency where one employee submits a portfolio, another submits a video link, and another submits a physical painting. You need a flexible filing cabinet that can store different types of folders without a strict format. That is MongoDB—it’s flexible, document-based, and scales easily when data types are unpredictable.

**P2:** Fair enough. One last question. Tell us about a time you had to manage a group of peers and things did not go smoothly.

**Candidate:** Recently, I planned and executed a 3-day group trip to Udaipur for my friends. What was supposed to be a relaxing weekend quickly became stressful when two people dropped out at the last minute, significantly impacting our per-head budget for the hotel and travel.

**P2:** How did you handle the deficit?

**Candidate:** I had to be transparent. I called a quick meeting, laid out the new numbers, and offered a choice: we either shorten the trip by one day, or we switch to a slightly more affordable hotel slightly further from Lake Pichola. I facilitated a quick anonymous vote to avoid peer pressure. We opted for the cheaper hotel. It taught me that when managing a team's expectations, transparent communication regarding resource constraints is better than trying to quietly fix it yourself.

**P1:** Alright. We are done here. Thank you, you may leave the room.

**Candidate:** Thank you, professors. Have a good day."""

In [50]:
import re

def clean_transcript(transcript: str) -> str:
    # Remove triple quotes if present
    transcript = transcript.strip().strip('"""').strip("'''")

    # Remove markdown headings
    transcript = re.sub(r'^#+\s*', '', transcript, flags=re.MULTILINE)

    # Remove bold markers
    transcript = transcript.replace('**', '')

    # Remove empty lines
    transcript = re.sub(r'\n\s*\n+', '\n', transcript)

    # Normalize whitespace
    transcript = re.sub(r'[ \t]+', ' ', transcript)

    return transcript.strip()

In [51]:
import re

def clean_transcript_keep_roles(transcript: str) -> str:
    # 1. Remove bold markdown
    transcript = transcript.replace('**', '')

    # 2. Normalize Interviewer labels and KEEP their unique IDs

    # Catch full names: "Interviewer 2 (P2):" or "Interviewer 3:" -> "INTERVIEWER 2:", "INTERVIEWER 3:"
    transcript = re.sub(
        r'Interviewer\s*(\d+)\s*(?:\(P\d+\))?:',
        r'INTERVIEWER \1:',
        transcript,
        flags=re.IGNORECASE
    )

    # Catch shorthand labels: "P2:" or "P3:" -> "INTERVIEWER 2:", "INTERVIEWER 3:"
    transcript = re.sub(
        r'P(\d+):',
        r'INTERVIEWER \1:',
        transcript,
        flags=re.IGNORECASE
    )

    # Catch alphabetical labels: "Interviewer B:" -> "INTERVIEWER B:"
    transcript = re.sub(
        r'Interviewer\s*([A-Za-z]):',
        r'INTERVIEWER \1:',
        transcript,
        flags=re.IGNORECASE
    )

    # 3. Normalize Candidate label
    transcript = re.sub(
        r'Candidate:',
        'CANDIDATE:',
        transcript,
        flags=re.IGNORECASE
    )

    # 4. Remove extra empty lines, keeping clean paragraph spacing
    transcript = re.sub(r'\n\s*\n+', '\n\n', transcript)

    return transcript.strip()

In [115]:
template = """
You are an experienced IIM interview panelist with extensive experience evaluating MBA applicants.

Analyze the interview transcript and evaluate the candidate as an admissions panel would.

Important Instructions:
* Evaluate only based on evidence present in the transcript.
* Do not assume traits, achievements, knowledge, or behaviors that were not demonstrated.
* Be highly critical and realistic.
* A score of 5-6 represents an average MBA applicant.
* A score of 7-8 represents a strong candidate.
* A score of 9 represents exceptional performance.
* A score of 10 should be reserved for rare, near-flawless responses.
* Most candidates should receive scores between 5 and 8.
* If a dimension was not sufficiently tested during the interview, assign null instead of guessing.
* Justify every score using specific evidence from the transcript.
* Interview difficulty should be medium to hard as in real interview evaluation.

Verdict Guidelines:
* Strong Hire: Consistently impressive across most dimensions with no major weaknesses.
* Lean Hire: Good candidate with some gaps but overall positive assessment.
* Lean Reject: Mixed performance with noticeable weaknesses.
* Strong Reject: Multiple critical weaknesses or inability to answer key questions.

Interview Transcript:
{transcript}
"""

In [116]:
template

'\nYou are an experienced IIM interview panelist with extensive experience evaluating MBA applicants.\n\nAnalyze the interview transcript and evaluate the candidate as an admissions panel would.\n\nImportant Instructions:\n* Evaluate only based on evidence present in the transcript.\n* Do not assume traits, achievements, knowledge, or behaviors that were not demonstrated.\n* Be highly critical and realistic.\n* A score of 5-6 represents an average MBA applicant.\n* A score of 7-8 represents a strong candidate.\n* A score of 9 represents exceptional performance.\n* A score of 10 should be reserved for rare, near-flawless responses.\n* Most candidates should receive scores between 5 and 8.\n* If a dimension was not sufficiently tested during the interview, assign null instead of guessing.\n* Justify every score using specific evidence from the transcript.\n* Interview difficulty should be medium to hard as in real interview evaluation.\n\nVerdict Guidelines:\n* Strong Hire: Consistently 

In [20]:
!pip install langchain langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.4 MB/s eta 0:00:00


In [176]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    # model="llama-3.1-8b-instant",
    api_key=GROQ_API_KEY,
    temperature=0.2,
    max_tokens=6000
)

In [118]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["transcript"],
    template=template
)

In [119]:
cleaned_transcript= clean_transcript_keep_roles(clean_transcript(transcript))

In [120]:
from typing import Optional, List, Literal
from pydantic import BaseModel, Field

class ScoreDimension(BaseModel):
  score: Optional[int] = Field(
  default=None,
  description="Score from 1-10. Null if not sufficiently evaluated."
  )
  evidence: str = Field(
  description="Specific evidence from the transcript supporting the score."
  )

class Scores(BaseModel):
  communication_skills: ScoreDimension
  academic_fundamentals: ScoreDimension
  analytical_thinking: ScoreDimension
  general_awareness: ScoreDimension
  mba_motivation: ScoreDimension
  career_clarity: ScoreDimension
  domain_expertise: ScoreDimension
  self_awareness: ScoreDimension
  leadership_potential: ScoreDimension

class InterviewEvaluation(BaseModel):
  overall_summary: str = Field(
    description="3-4 sentence holistic assessment of the candidate."
  )

  overall_score: float = Field(
        description="Overall interview score out of 10."
    )

  strengths: List[str] = Field(
      description="Specific demonstrated strengths."
  )

  weaknesses: List[str] = Field(
      description="Specific demonstrated weaknesses."
  )

  panel_concerns: List[str] = Field(
      description="Potential concerns for the admissions committee."
  )

  # interview_difficulty: Literal[
  #     "Easy",
  #     "Moderate",
  #     "Difficult"
  # ]

  scores: Scores

  final_verdict: Literal[
      "Strong Hire",
      "Lean Hire",
      "Lean Reject",
      "Strong Reject"
  ]

  feedback_to_candidate: str = Field(
      description="Actionable feedback for future MBA interviews."
  )


In [121]:
structured_llm = llm.with_structured_output(InterviewEvaluation)

In [122]:
chain = prompt | structured_llm

In [123]:
result=chain.invoke(cleaned_transcript)

In [124]:
print(result.final_verdict)
print(result.scores.mba_motivation.score)
print(result.scores.mba_motivation.evidence)

Lean Hire
8
The candidate was able to articulate his motivations for pursuing an MBA.


In [146]:
from IPython.display import display, Markdown

def display_interview_report(result):
    md = f"""
# 🎓 IIM Interview Evaluation Report

## 📊 Overall Assessment
* **Overall Score:** **{result.overall_score:.1f} / 10**
* **Final Verdict:** **{result.final_verdict}**

### 📝 Summary
> {result.overall_summary}

### 💪 Strengths
"""
    for strength in result.strengths:
        md += f"* {strength}\n"

    md += "\n### 📉 Areas for Improvement\n"
    for weakness in result.weaknesses:
        md += f"* {weakness}\n"

    md += "\n### ⚠️ Panel Concerns\n"
    for concern in result.panel_concerns:
        md += f"* {concern}\n"

    # --- UPDATED HTML TABLE: Right-aligned Scores and Evidence ---
    md += """
## 🎯 Dimension-wise Scores

<table style="width: 100%; border-collapse: collapse;">
  <tr style="border-bottom: 2px solid #888;">
    <th style="text-align: left; padding: 8px; width: 20%;">Dimension</th>
    <th style="text-align: right; padding: 8px; width: 10%;">Score</th>
    <th style="text-align: right; padding: 8px; width: 70%;">Evidence</th>
  </tr>
"""
    score_dict = result.scores.model_dump()

    for dimension, data in score_dict.items():
        score = data.get("score")
        evidence = data.get("evidence")

        score_text = "N/A" if score is None else f"<b>{score}/10</b>"
        dimension_formatted = dimension.replace('_', ' ').title()

        # Notice text-align: right on the second two <td> tags
        md += f"""
  <tr style="border-bottom: 1px solid #555;">
    <td style="text-align: left; padding: 8px;"><b>{dimension_formatted}</b></td>
    <td style="text-align: right; padding: 8px;">{score_text}</td>
    <td style="text-align: right; padding: 8px;">{evidence}</td>
  </tr>
"""

    md += "</table>\n"

    md += f"""
---
## 💡 Feedback to Candidate
{result.feedback_to_candidate}
"""

    display(Markdown(md))

In [147]:
display_interview_report(result=result)



# 🎓 IIM Interview Evaluation Report

## 📊 Overall Assessment
* **Overall Score:** **7.2 / 10**
* **Final Verdict:** **Lean Hire**

### 📝 Summary
> Priyanshu is a strong candidate with a solid foundation in technical skills, but he needs to work on his communication and leadership skills to become a more well-rounded MBA applicant. He demonstrated a clear understanding of business acumen and macro-level awareness, but struggled to provide a clear, concise answer to the question about managing a group of peers. With further development, Priyanshu has the potential to become a strong MBA candidate.

### 💪 Strengths
* Technical skills
* Business acumen
* Macro-level awareness

### 📉 Areas for Improvement
* Communication skills
* Leadership skills

### ⚠️ Panel Concerns
* Communication skills
* Leadership skills

## 🎯 Dimension-wise Scores

<table style="width: 100%; border-collapse: collapse;">
  <tr style="border-bottom: 2px solid #888;">
    <th style="text-align: left; padding: 8px; width: 20%;">Dimension</th>
    <th style="text-align: right; padding: 8px; width: 10%;">Score</th>
    <th style="text-align: right; padding: 8px; width: 70%;">Evidence</th>
  </tr>

  <tr style="border-bottom: 1px solid #555;">
    <td style="text-align: left; padding: 8px;"><b>Communication Skills</b></td>
    <td style="text-align: right; padding: 8px;"><b>5/10</b></td>
    <td style="text-align: right; padding: 8px;">The candidate struggled to provide a clear, concise answer to the question about managing a group of peers.</td>
  </tr>

  <tr style="border-bottom: 1px solid #555;">
    <td style="text-align: left; padding: 8px;"><b>Academic Fundamentals</b></td>
    <td style="text-align: right; padding: 8px;"><b>8/10</b></td>
    <td style="text-align: right; padding: 8px;">The candidate was able to explain the difference between a relational database like PostgreSQL and a non-relational one like MongoDB using a business analogy.</td>
  </tr>

  <tr style="border-bottom: 1px solid #555;">
    <td style="text-align: left; padding: 8px;"><b>Analytical Thinking</b></td>
    <td style="text-align: right; padding: 8px;"><b>8/10</b></td>
    <td style="text-align: right; padding: 8px;">The candidate was able to analyze the situation with his father's business and come up with two immediate strategies to increase profit margins.</td>
  </tr>

  <tr style="border-bottom: 1px solid #555;">
    <td style="text-align: left; padding: 8px;"><b>General Awareness</b></td>
    <td style="text-align: right; padding: 8px;"><b>8/10</b></td>
    <td style="text-align: right; padding: 8px;">The candidate demonstrated a good understanding of the current business environment and trends.</td>
  </tr>

  <tr style="border-bottom: 1px solid #555;">
    <td style="text-align: left; padding: 8px;"><b>Mba Motivation</b></td>
    <td style="text-align: right; padding: 8px;"><b>8/10</b></td>
    <td style="text-align: right; padding: 8px;">The candidate was able to articulate his motivations for pursuing an MBA.</td>
  </tr>

  <tr style="border-bottom: 1px solid #555;">
    <td style="text-align: left; padding: 8px;"><b>Career Clarity</b></td>
    <td style="text-align: right; padding: 8px;"><b>8/10</b></td>
    <td style="text-align: right; padding: 8px;">The candidate was able to articulate his career goals and motivations for pursuing an MBA.</td>
  </tr>

  <tr style="border-bottom: 1px solid #555;">
    <td style="text-align: left; padding: 8px;"><b>Domain Expertise</b></td>
    <td style="text-align: right; padding: 8px;"><b>8/10</b></td>
    <td style="text-align: right; padding: 8px;">The candidate demonstrated a good understanding of the automotive sector and the Total Cost of Ownership (TCO) of diesel engines.</td>
  </tr>

  <tr style="border-bottom: 1px solid #555;">
    <td style="text-align: left; padding: 8px;"><b>Self Awareness</b></td>
    <td style="text-align: right; padding: 8px;"><b>8/10</b></td>
    <td style="text-align: right; padding: 8px;">The candidate demonstrated a good understanding of his strengths and weaknesses.</td>
  </tr>

  <tr style="border-bottom: 1px solid #555;">
    <td style="text-align: left; padding: 8px;"><b>Leadership Potential</b></td>
    <td style="text-align: right; padding: 8px;"><b>5/10</b></td>
    <td style="text-align: right; padding: 8px;">The candidate struggled to provide a clear, concise answer to the question about managing a group of peers.</td>
  </tr>
</table>

---
## 💡 Feedback to Candidate
Priyanshu demonstrated a clear understanding of business acumen and macro-level awareness, showcasing his ability to bridge technical skills with formal frameworks in finance and strategy. However, he struggled to provide a clear, concise answer to the question about managing a group of peers, and his response could have been more detailed. Overall, Priyanshu has a solid foundation in technical skills, but his ability to apply these skills in a business context is still developing. He would benefit from further development in his communication and leadership skills.


In [153]:
current_report=result.model_dump_json()

In [154]:
template_feedback="""You are an experienced IIM admissions mentor.

You have received a detailed interview evaluation report.

Your task is not to re-score the candidate.

Instead:

1. Identify the most important weaknesses.
2. Generate 5 realistic follow-up interview questions.
3. Explain why each question would likely be asked.
4. Provide an ideal MBA-level answer.
5. Create a personalized mock interview improvement plan.
6. Prioritize improvements that would most increase admission chances.
Interview Report: {current_report}
Return structured JSON.
"""

In [165]:
from typing import List, Optional, Literal
from pydantic import BaseModel, Field

# Modified FollowUpQuestion for generating just the questions first
class FollowUpQuestion(BaseModel):
  question: str = Field(description="The realistic follow-up interview question.")
  why_this_question: str = Field(description="Explanation of why this question would likely be asked, based on the candidate's weaknesses or areas for deeper probing.")

# New model to combine questions with their ideal MBA-level answers
class QuestionAnswerPair(BaseModel):
  question: str = Field(description="The realistic follow-up interview question.")
  why_this_question: str = Field(description="Explanation of why this question would likely be asked, based on the candidate's weaknesses or areas for deeper probing.")
  ideal_answer: str = Field(description="An ideal MBA-level answer for the follow-up question.")

class MockInterviewPlan(BaseModel):
  focus_area: str = Field(description="The specific area of improvement for the mock interview (e.g., 'Behavioral Questions', 'Strategic Thinking').")
  exercises: List[str] = Field(description="A list of specific exercises or practice techniques to improve the focus area (e.g., 'Practice STAR method for behavioral questions', 'Read HBR articles on business strategy').")

class InterviewCoachReport(BaseModel):
  key_weaknesses_to_improve: List[str] = Field(description="A list of the most important weaknesses identified from the interview evaluation that the candidate needs to address.")
  follow_up_questions: List[QuestionAnswerPair] = Field(description="A list of realistic follow-up interview questions with explanations and ideal MBA-level answers.")
  mock_interview_plan: List[MockInterviewPlan] = Field(description="A personalized plan for mock interviews, outlining focus areas and specific exercises.")
  priority_actions: List[str] = Field(description="A list of 1-2 concise, high-impact actions the candidate should prioritize to increase admission chances.")
  readiness_assessment: str = Field(description="An overall assessment of the candidate's current readiness for MBA interviews and what further steps are needed.")

In [161]:
feedback_llm = llm.with_structured_output(InterviewCoachReport)

In [162]:

feedback_prompt = PromptTemplate(
    input_variables=["current_report"],
    template=template_feedback
)

In [163]:
feedback_chain= feedback_prompt | feedback_llm

In [164]:
feedback_chain.invoke(current_report)

APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `llama-3.1-8b-instant` in organization `org_01kfxvqa77fbqb05kjb1cjze03` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 7010, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [177]:
optimized_feedback_input = {
    "overall_score": result.overall_score,
    "weaknesses": result.weaknesses,
    "panel_concerns": result.panel_concerns,
    "dimension_scores": result.scores.model_dump() # Convert the Pydantic Scores object to a dictionary
}

print(optimized_feedback_input)

{'weaknesses': ['Communication skills', 'Leadership skills'], 'panel_concerns': ['Communication skills', 'Leadership skills'], 'dimension_scores': {'communication_skills': {'score': 5, 'evidence': 'The candidate struggled to provide a clear, concise answer to the question about managing a group of peers.'}, 'academic_fundamentals': {'score': 8, 'evidence': 'The candidate was able to explain the difference between a relational database like PostgreSQL and a non-relational one like MongoDB using a business analogy.'}, 'analytical_thinking': {'score': 8, 'evidence': "The candidate was able to analyze the situation with his father's business and come up with two immediate strategies to increase profit margins."}, 'general_awareness': {'score': 8, 'evidence': 'The candidate demonstrated a good understanding of the current business environment and trends.'}, 'mba_motivation': {'score': 8, 'evidence': 'The candidate was able to articulate his motivations for pursuing an MBA.'}, 'career_clarit

In [180]:
from typing import List
from pydantic import BaseModel, Field

# Assuming FollowUpQuestion is defined in a previous cell, re-defining it here for completeness
class FollowUpQuestion(BaseModel):
  question: str = Field(description="The realistic follow-up interview question.")
  why_this_question: str = Field(description="Explanation of why this question would likely be asked, based on the candidate's weaknesses or areas for deeper probing.")

# Define a container model for the list of follow-up questions
class FollowUpQuestionsContainer(BaseModel):
    questions: List[FollowUpQuestion] = Field(description="A list of generated follow-up interview questions.")

follow_up_questions_template = """You are an experienced IIM admissions mentor. You have reviewed the following candidate's initial interview evaluation.

Based on the identified weaknesses, panel concerns, and dimension scores, your task is to generate 5 realistic follow-up interview questions.

For each question, explain why it would likely be asked to further probe the candidate's capabilities or address the identified areas for improvement.

Interview Evaluation Summary: {{optimized_feedback_input}}
"""

follow_up_questions_prompt = PromptTemplate(
    input_variables=["optimized_feedback_input"],
    template=follow_up_questions_template
)

# Use the new container model here
structured_follow_up_llm = llm.with_structured_output(FollowUpQuestionsContainer)

follow_up_questions_chain = follow_up_questions_prompt | structured_follow_up_llm

# Invoke and access the 'questions' field from the container
follow_up_questions_output = follow_up_questions_chain.invoke(optimized_feedback_input)
follow_up_questions_list = follow_up_questions_output.questions

print(follow_up_questions_list)

[FollowUpQuestion(question='Can you provide a specific example of a time when you overcame a difficult challenge in your previous role?', why_this_question="To assess the candidate's ability to handle pressure and demonstrate resilience, as their initial interview evaluation highlighted concerns about their stress management skills."), FollowUpQuestion(question='How do you prioritize tasks and manage your time when working on multiple projects simultaneously?', why_this_question="To evaluate the candidate's organizational skills and ability to multitask, as their dimension scores indicated a need for improvement in this area."), FollowUpQuestion(question='Can you walk us through your decision-making process when faced with a complex problem?', why_this_question="To further probe the candidate's critical thinking and problem-solving skills, as the panel expressed concerns about their ability to think strategically."), FollowUpQuestion(question='How do you handle feedback or constructive

In [183]:
class IdealAnswer(BaseModel):
    question: str = Field(
        description="The interview question being answered."
    )

    ideal_answer: str = Field(
        description="An ideal MBA interview answer."
    )

    key_points: list[str] = Field(
        description="Important points demonstrated in the answer."
    )

In [182]:
answer_template = """
You are an IIM admissions mentor.

Generate an ideal MBA interview answer for the question below.

The answer should:

- be concise
- be realistic
- demonstrate strong communication
- demonstrate structured thinking
- reflect IIM-level expectations

Question:

{question}
"""

In [186]:
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate

answer_prompt = PromptTemplate(
    input_variables=["question"],
    template=answer_template
)

In [184]:
structured_answer_llm = llm.with_structured_output(
    IdealAnswer
)

In [187]:
answer_chain = answer_prompt | structured_answer_llm


In [188]:
answers = []

for q in follow_up_questions_list:
    answer = answer_chain.invoke(
        {"question": q.question}
    )
    answers.append(answer)

In [189]:
for i, ans in enumerate(answers, start=1):
    print(f"\n{'='*80}")
    print(f"QUESTION {i}")
    print(f"{'='*80}")

    print("\nQuestion:")
    print(ans.question)

    print("\nIdeal Answer:")
    print(ans.ideal_answer)

    print("\nKey Points:")
    for point in ans.key_points:
        print(f"• {point}")


QUESTION 1

Question:
Can you provide a specific example of a time when you overcame a difficult challenge in your previous role?

Ideal Answer:
In my previous role as a marketing manager, I faced a challenging situation where our team's campaign was underperforming, and we were at risk of missing our quarterly targets. I took the lead, analyzed the data, and identified the root cause of the issue. I then collaborated with the team to develop a revised strategy, which included adjusting our targeting and messaging. We successfully implemented the new plan, and our campaign's performance improved significantly, exceeding our targets by 15%. This experience taught me the importance of data-driven decision-making, effective teamwork, and adaptability in overcoming challenges.

Key Points:
• Identifying the root cause of the problem
• Collaborating with the team to develop a revised strategy
• Implementing a new plan and achieving positive results

QUESTION 2

Question:
How do you priorit

In [190]:
coach_output = {
    "follow_up_questions": [
        q.model_dump() for q in follow_up_questions_list
    ],
    "ideal_answers": [
        a.model_dump() for a in answers
    ]
}

In [191]:
# Past Reports (WINDOW_SIZE = 4)
previous_reports = [
    {
        "overall_score": 5.0,
        "strengths": [
            "Strong academic record in engineering",
            "Polite and respectful demeanor"
        ],
        "weaknesses": [
            "Extremely vague goals and justification for pursuing an MBA",
            "Lacks basic awareness of current macroeconomic events",
            "Answers are long-winded and lack a clear bottom-line"
        ],
        "panel_concerns": [
            "Candidate seems to be doing an MBA purely as a default next step.",
            "Struggled to name a single recent business news story."
        ],
        "feedback_to_candidate": "You need to build a compelling narrative for 'Why MBA'. Start reading a pink paper (business newspaper) daily. Practice structuring your answers to avoid rambling."
    },
    {
        "overall_score": 6.0,
        "strengths": [
            "Solid revision of basic undergraduate technical concepts",
            "Noticeable improvement in communication clarity"
        ],
        "weaknesses": [
            "The 'Why MBA' answer still feels superficial and rehearsed",
            "Struggles to structure answers under sudden cross-questioning",
            "Surface-level understanding of banking sector reforms"
        ],
        "panel_concerns": [
            "Becomes visibly nervous when pushed out of their comfort zone.",
            "Relies heavily on buzzwords instead of deep analytical points."
        ],
        "feedback_to_candidate": "Good effort on your communication. Now, focus on structuring your thoughts using frameworks (like STAR). Deepen your general knowledge—don't just read headlines."
    },
    {
        "overall_score": 7.0,
        "strengths": [
            "Excellent framework-driven answer structure",
            "Good understanding of current tech-industry trends and AI regulations"
        ],
        "weaknesses": [
            "Still hesitates slightly when grilled on the 'Why MBA' shift from engineering",
            "Body language becomes rigid under heavy stress testing"
        ],
        "panel_concerns": [
            "While structure has improved, spontaneity is missing.",
            "Anxiety leaks through when the panel disagrees with their points."
        ],
        "feedback_to_candidate": "Your answer structure is miles ahead of where you started! Keep that up. To clear the final hurdle, work on your composure during stress testing. Accept alternative viewpoints gracefully."
    },
    {
        "overall_score": 6.5,
        "strengths": [
            "Very strong grasp of recent Union Budget highlights and geopolitical events",
            "Clear, concise delivery in HR questions"
        ],
        "weaknesses": [
            "Froze on a fundamental undergrad mathematics question",
            "The career transition narrative ('Why MBA') still lacks absolute conviction"
        ],
        "panel_concerns": [
            "Candidate has started neglecting their core academic discipline.",
            "Core motivation for management still feels slightly shaky under pressure."
        ],
        "feedback_to_candidate": "Your current affairs knowledge was highly impressive today. However, you cannot afford to forget your undergraduate fundamentals. Re-verify your academic basics and plug the gaps in your MBA story."
    }
]

# Current Report
historical_snapshot = {
    "overall_score": 7.8,
    "strengths": [
        "Exceptional composure and confidence during a prolonged stress interview",
        "Deep, analytical breakdown of complex macroeconomic issues",
        "Highly structured and precise delivery"
    ],
    "weaknesses": [
        "Completely failed to answer core undergraduate engineering questions",
        "The long-term career goal answer remains a bit generic"
    ],
    "panel_concerns": [
        "A clear blindspot has developed regarding undergrad subjects; it seems completely deprioritized.",
        "Struggles to articulate a highly specific post-MBA career path beyond 'Consulting'."
    ],
    "feedback_to_candidate": "Phenomenal growth in your delivery, structure, and current affairs. You handled our cross-questioning beautifully today. Your absolute top priority now must be brushing up on your engineering basics—the panel will punish you if you forget your roots."
}

In [192]:
from typing import List
from pydantic import BaseModel, Field

class ProgressAnalysis(BaseModel):

    improved_areas: List[str] = Field(
        description="Areas showing improvement across interviews."
    )

    declining_areas: List[str] = Field(
        description="Areas that became weaker."
    )

    recurring_weaknesses: List[str] = Field(
        description="Weaknesses appearing repeatedly."
    )

    growth_summary: str = Field(
        description="Overall growth trend."
    )

    next_focus_areas: List[str] = Field(
        description="Highest priority improvement areas."
    )

In [193]:
progress_input = {
    "historical_reports": previous_reports,
    "current_report": historical_snapshot
}

In [194]:
progress_template = """
You are an experienced IIM admissions mentor.

You are given summaries of a candidate's past interview reports
and the latest interview report.

Analyze the candidate's progress over time.

Tasks:

1. Identify areas that improved.
2. Identify areas that declined.
3. Identify recurring weaknesses.
4. Summarize overall growth.
5. Recommend next focus areas.

Focus only on trends visible across reports.

Past Reports:
{historical_reports}

Current Report:
{current_report}
"""

In [195]:
progress_prompt = PromptTemplate(
    input_variables=[
        "historical_reports",
        "current_report"
    ],
    template=progress_template
)

structured_progress_llm = llm.with_structured_output(
    ProgressAnalysis
)

progress_chain = (
    progress_prompt
    | structured_progress_llm
)

In [196]:
import json

progress_output = progress_chain.invoke(
    {
        "historical_reports": json.dumps(
            previous_reports,
            indent=2
        ),
        "current_report": json.dumps(
            historical_snapshot,
            indent=2
        )
    }
)

In [197]:
def display_progress_report(analysis: ProgressAnalysis):
    md_content = f"""# 📈 Candidate Progress Analysis

### 🌟 Improved Areas
{chr(10).join([f"- {item}" for item in analysis.improved_areas]) if analysis.improved_areas else "- None identified"}

### 📉 Declining Areas
{chr(10).join([f"- {item}" for item in analysis.declining_areas]) if analysis.declining_areas else "- None identified"}

### 🔄 Recurring Weaknesses
{chr(10).join([f"- {item}" for item in analysis.recurring_weaknesses]) if analysis.recurring_weaknesses else "- None identified"}

### 📊 Growth Summary
{analysis.growth_summary}

### 🎯 Next Focus Areas
{chr(10).join([f"{i+1}. {item}" for i, item in enumerate(analysis.next_focus_areas)]) if analysis.next_focus_areas else "- None identified"}
"""
    # 3. Render it in the notebook
    display(Markdown(md_content))


In [198]:
display_progress_report(progress_output)

# 📈 Candidate Progress Analysis

### 🌟 Improved Areas
- communication clarity
- answer structure
- current affairs knowledge
- composure under stress

### 📉 Declining Areas
- undergraduate academic fundamentals
- conviction in career transition narrative

### 🔄 Recurring Weaknesses
- lack of absolute conviction in 'Why MBA' answer
- difficulty in articulating a specific post-MBA career path

### 📊 Growth Summary
The candidate has shown significant improvement in communication skills, answer structure, and current affairs knowledge, but has neglected their undergraduate academic fundamentals and still struggles with conviction in their career transition narrative.

### 🎯 Next Focus Areas
1. revising undergraduate engineering basics
2. developing a more specific post-MBA career path
3. strengthening the 'Why MBA' narrative
